# 02 — Dynamic Layers

Every map we have built adds layers at cell-run time and leaves them there. This lesson makes layers **runtime objects** — created, swapped, cleared, and toggled while the map is live.

This matters because real analysis workflows are not static: you load a dataset, filter it, show a subset, update the filter, swap the view. All of that happens by manipulating layers on a live map.

## What Is a Layer?

A map is a stack of independent layers rendered on top of each other:

```
Map
 ├── Base tile layer     (background imagery / street map)
 ├── GeoJSON layer       (polygons, lines, points from a file)
 ├── Marker layer        (individual pins)
 └── ... any number more
```

Each layer is a separate Python object. You can:

- `m.add(layer)` — put it on the map
- `m.remove(layer)` — take it off
- Mutate the layer object — the map reflects the change immediately

The map is a **stateful system**: it remembers what layers are on it, and changes to any layer propagate to the display instantly.

## Setup

In [ ]:
import json
from ipyleaflet import Map, GeoJSON, Marker, Polyline

WICHITA_FALLS = (33.9137, -98.4934)

with open("../02-Viewing_GeoJSON/data/wichita_falls.geojson") as f:
    geojson = json.load(f)

def make_fc(features):
    return {"type": "FeatureCollection", "features": features}

## Adding and Removing a Layer

Keep a reference to the layer object. `m.add()` puts it on the map; `m.remove()` takes it off. Run the cells in sequence and watch the map update.

In [ ]:
m = Map(center=WICHITA_FALLS, zoom=12)

park_layer = GeoJSON(
    data=make_fc([f for f in geojson["features"] if f["properties"].get("type") == "park"]),
    style={"color": "#27ae60", "fillColor": "#27ae60", "fillOpacity": 0.45, "weight": 2}
)

m.add(park_layer)
m

In [ ]:
# Remove the layer — map above updates immediately
m.remove(park_layer)

In [ ]:
# Add it back
m.add(park_layer)

## Tracking Layers in a Dict

When you have multiple named layers, a dict is the cleanest way to keep track of them. Keys are names; values are layer objects.

In [ ]:
COLOR_MAP = {
    "park":       "#27ae60",
    "water":      "#1abc9c",
    "education":  "#3498db",
    "government": "#e74c3c",
    "route":      "#e67e22",
}

# Build one GeoJSON layer per feature type
layers = {}
for feat_type, color in COLOR_MAP.items():
    features = [f for f in geojson["features"] if f["properties"].get("type") == feat_type]
    if features:
        layers[feat_type] = GeoJSON(
            data=make_fc(features),
            name=feat_type.capitalize(),
            style={"color": color, "fillColor": color, "fillOpacity": 0.45, "weight": 2}
        )

print("Layers built:", list(layers.keys()))

m = Map(center=WICHITA_FALLS, zoom=12)
for layer in layers.values():
    m.add(layer)
m

## Clearing Layers

To remove all non-base layers at once, iterate the dict and call `m.remove()` on each.

In [ ]:
# Clear every layer tracked in the dict
for layer in layers.values():
    try:
        m.remove(layer)
    except Exception:
        pass  # already removed — safe to ignore

print("All layers removed")

In [ ]:
# Restore them all
for layer in layers.values():
    m.add(layer)

print("All layers restored")

## Toggling a Single Layer

Show/hide one layer without touching the others. The pattern: check whether the layer is currently on the map, then add or remove accordingly.

ipyleaflet exposes the current layers as `m.layers` — a tuple of all layer objects currently attached to the map.

In [ ]:
def toggle_layer(map_obj, layer):
    """Add layer if absent, remove it if present."""
    if layer in map_obj.layers:
        map_obj.remove(layer)
        print(f"Removed: {layer.name}")
    else:
        map_obj.add(layer)
        print(f"Added:   {layer.name}")

# Run this cell repeatedly — it flips the park layer on and off
toggle_layer(m, layers["park"])

## Swapping Layers

A common pattern: show one layer, hide the rest. Useful for switching between views of the same area.

In [ ]:
def show_only(map_obj, layer_dict, key):
    """Show one layer by key, hide all others in the dict."""
    for name, layer in layer_dict.items():
        if name == key:
            if layer not in map_obj.layers:
                map_obj.add(layer)
        else:
            if layer in map_obj.layers:
                map_obj.remove(layer)
    print(f"Showing only: {key}")

# Show only water features
show_only(m, layers, "water")

In [ ]:
# Switch to education features
show_only(m, layers, "education")

In [ ]:
# Restore all layers
for layer in layers.values():
    if layer not in m.layers:
        m.add(layer)

## Click-to-Place with a Clear Button

Combining everything: click the map to drop markers, click a button to wipe them all. `ipywidgets.Button` fires a callback when pressed — the same observer pattern as `on_interaction`.

In [ ]:
from ipywidgets import Button, VBox

m = Map(center=WICHITA_FALLS, zoom=12)
placed_markers = []

def on_click(**kwargs):
    if kwargs.get("type") != "click":
        return
    lat, lon = kwargs["coordinates"]
    marker = Marker(location=(lat, lon), title=f"Point {len(placed_markers) + 1}")
    placed_markers.append(marker)
    m.add(marker)
    print(f"[{len(placed_markers)}]  ({lat:.5f}, {lon:.5f})")

def on_clear(btn):
    for marker in placed_markers:
        m.remove(marker)
    placed_markers.clear()
    print("Cleared")

m.on_interaction(on_click)

clear_btn = Button(description="Clear Markers", button_style="danger")
clear_btn.on_click(on_clear)

VBox([m, clear_btn])

Click the map to place markers. Click **Clear Markers** to remove them all at once.

Two things to notice:
- `placed_markers` tracks the layer objects so we know what to remove
- `placed_markers.clear()` resets the list so the count starts from 1 again on the next round

## Exercise A

Build a click counter tool: every click on the map adds a marker and increments a `Label` widget showing `"Clicks: N"`. A **Reset** button zeroes the count and removes all placed markers.

The starter code below is complete — study it, then modify it to also print each click's coordinates to an `Output` widget instead of the default cell output.

In [ ]:
from ipyleaflet import Map, Marker
from ipywidgets import Label, Button, HBox, VBox

m = Map(center=WICHITA_FALLS, zoom=12)
click_count = [0]
click_label = Label(value="Clicks: 0")
placed = []

def on_click(**kwargs):
    if kwargs.get("type") != "click":
        return
    lat, lon = kwargs["coordinates"]
    click_count[0] += 1
    click_label.value = f"Clicks: {click_count[0]}"
    marker = Marker(location=(lat, lon))
    placed.append(marker)
    m.add(marker)

def on_reset(btn):
    for marker in placed:
        m.remove(marker)
    placed.clear()
    click_count[0] = 0
    click_label.value = "Clicks: 0"

m.on_interaction(on_click)

reset_btn = Button(description="Reset", button_style="danger")
reset_btn.on_click(on_reset)

VBox([m, HBox([click_label, reset_btn])])

## Exercise B

Build a button panel with one `Button` per feature type (park, water, education, government, route). Clicking a button calls `show_only()` to display only that layer. Below the map, an `Output` widget updates to show which layer is currently visible.

In [ ]:
from ipywidgets import Button, HBox, VBox, Output

m2 = Map(center=WICHITA_FALLS, zoom=12)
for layer in layers.values():
    m2.add(layer)

layer_status = Output(layout={"border": "1px solid #ccc", "padding": "6px"})

# Build one Button per feature type; clicking it calls show_only(m2, layers, type_name)
# After each click, update layer_status to show which layers are currently visible
# Your code here

VBox([m2, layer_status])

---

## Check Your Understanding

Build a map with two GeoJSON layers from `wichita_falls.geojson` — one for Points, one for Polygons. Add two `Button` widgets: **"Show Points Only"** and **"Show Both"**. Each button should update the map accordingly.

```python
# your answer here
```

## Next

In [03 — User Feedback](./03-User_Feedback.ipynb), we add output widgets, status panels, and computed results that display alongside the map as the user interacts.